# ML models

Train six classifiers on four feature tables from `01_clean_dataset.ipynb`:

1. **Baseline** — IEEE-CIS columns only (no `uid`, `uid2`, or `DT_*`)
2. **Feature Engineering** — baseline plus `uid`, `uid2`, and `DT_*`
3. **Reduced Baseline** — Table 3 filters (≥90% empty, |r| > 0.98, IG < 0.001), no `uid` / `uid2` / `DT_*`
4. **Reduced Feature Engineering** — reduced table plus `uid` / `uid2` / `DT_*` if they survived the filters

Model order: logistic regression, decision tree, random forest, LightGBM, XGBoost, CatBoost.

Hyperparameters are tuned with **Optuna** (TPE, maximize ROC-AUC) on an inner temporal split of the 80% train set. The original 20% holdout is scored only after the best params are refit. Set `USE_OPTUNA = False` or `N_TRIALS = 0` to skip tuning.

Each run is appended to `saved/ml_results.parquet` (including `BestParams` and inner `TuneROC-AUC`). Charts live in `02_ml_models_result.ipynb`.

Kernel: `ai`.


In [1]:
import json
import warnings

import numpy as np
import pandas as pd
import altair as alt
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")
%matplotlib inline
pd.options.display.precision = 15
alt.renderers.enable("mimetype")
%env JOBLIB_TEMP_FOLDER=/tmp


env: JOBLIB_TEMP_FOLDER=/tmp


In [2]:
from pathlib import Path
import pandas as pd

try:
    import google.colab

    IS_COLAB = True
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    IS_COLAB = False

ROOT = Path("/content/drive/MyDrive/minor-thesis") if IS_COLAB else Path.cwd()
DATASET_PATH = ROOT / "dataset"
SAVED_PATH = ROOT / "saved"
SAVED_PATH.mkdir(parents=True, exist_ok=True)

print(f"Running on {'Google Colab' if IS_COLAB else 'Local'}")
print(f"Dataset path: {DATASET_PATH}")


Running on Local
Dataset path: d:\source\RMIT\master-of-ai-new\2026-semester-02\minor-thesis\dataset


In [3]:
train = pd.read_parquet(f"{DATASET_PATH}/merged_train.parquet")
test = pd.read_parquet(f"{DATASET_PATH}/merged_test.parquet")

RANDOM_SEED = 42
START_DATE = "2026-01-01"
print(f"train {train.shape}  test {test.shape}")


train (590540, 441)  test (506691, 440)


In [4]:
import gc
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier


def unwrap_estimator(model):
    if hasattr(model, "named_steps"):
        return list(model.named_steps.values())[-1]
    if hasattr(model, "estimator"):
        return model.estimator
    return model


def top_feature_importances(model, columns, n=20):
    est = unwrap_estimator(model)
    if hasattr(est, "calibrated_classifiers_"):
        inner = est.calibrated_classifiers_[0]
        est = getattr(inner, "estimator", getattr(inner, "base_estimator", est))
    if hasattr(est, "feature_importances_"):
        importances = np.asarray(est.feature_importances_, dtype=float)
    elif hasattr(est, "coef_"):
        importances = np.abs(np.asarray(est.coef_, dtype=float).ravel())
    else:
        return []
    if importances.size != len(columns):
        return []
    total = float(importances.sum())
    n = min(int(n), len(importances))
    order = np.argsort(importances)[::-1][:n]
    rows = []
    for rank, idx in enumerate(order, start=1):
        value = float(importances[idx])
        pct = (value / total * 100.0) if total > 0 else 0.0
        rows.append(
            {
                "rank": int(rank),
                "feature": str(columns[idx]),
                "importance": value,
                "importance_pct": pct,
            }
        )
    return rows


def positive_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(X)
    return model.predict(X).astype(float)


def evaluate(model, X_train, X_valid, y_train, y_valid, name):
    print(name)
    print(f"Features: {X_train.shape[1]}")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_valid)
    y_score = positive_scores(model, X_valid)
    accuracy = accuracy_score(y_valid, y_pred)
    precision = precision_score(y_valid, y_pred, zero_division=0)
    recall = recall_score(y_valid, y_pred, zero_division=0)
    f1 = f1_score(y_valid, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_valid, y_score)
    pr_auc = average_precision_score(y_valid, y_score)
    balanced_acc = balanced_accuracy_score(y_valid, y_pred)
    mcc = matthews_corrcoef(y_valid, y_pred)
    cm = confusion_matrix(y_valid, y_pred)
    print(f"Accuracy           : {accuracy:.4f}")
    print(f"Precision          : {precision:.4f}")
    print(f"Recall             : {recall:.4f}")
    print(f"F1 Score           : {f1:.4f}")
    print(f"ROC-AUC            : {roc_auc:.4f}")
    print(f"PR-AUC             : {pr_auc:.4f}")
    print(f"Balanced Accuracy  : {balanced_acc:.4f}")
    print(f"MCC                : {mcc:.4f}")
    print("\nConfusion Matrix:")
    print(cm)
    print("\nClassification Report:")
    print(
        classification_report(
            y_valid, y_pred, target_names=["Legitimate", "Fraud"], digits=4, zero_division=0
        )
    )
    top20 = top_feature_importances(model, X_train.columns)
    if top20:
        print("\nTop 20 feature importances:")
        for row in top20:
            print(
                f"  {row['rank']:2d}. {row['feature']:<32s} "
                f"{row['importance']:.6f}  ({row['importance_pct']:.2f}%)"
            )
    gc.collect()
    return {
        "Model": name,
        "Features": int(X_train.shape[1]),
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
        "Balanced Accuracy": balanced_acc,
        "MCC": mcc,
        "TN": int(cm[0, 0]),
        "FP": int(cm[0, 1]),
        "FN": int(cm[1, 0]),
        "TP": int(cm[1, 1]),
        "Top20Importances": json.dumps(top20),
    }


In [5]:
def make_logreg():
    return Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "clf",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=1000,
                    solver="lbfgs",
                    random_state=RANDOM_SEED,
                ),
            ),
        ]
    )


def make_decision_tree():
    return DecisionTreeClassifier(
        class_weight="balanced", random_state=RANDOM_SEED, max_depth=20
    )


def make_random_forest():
    return RandomForestClassifier(
        n_estimators=300, class_weight="balanced", random_state=RANDOM_SEED, n_jobs=-1
    )


def make_lightgbm():
    return LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=-1,
        num_leaves=31,
        class_weight="balanced",
        random_state=RANDOM_SEED,
        n_jobs=-1,
        verbosity=-1,
    )


def make_xgboost(y_train):
    n_neg = int((y_train == 0).sum())
    n_pos = max(int((y_train == 1).sum()), 1)
    return XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        random_state=RANDOM_SEED,
        n_jobs=-1,
        eval_metric="logloss",
        scale_pos_weight=n_neg / n_pos,
    )


def make_catboost():
    return CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        random_seed=RANDOM_SEED,
        auto_class_weights="Balanced",
        verbose=False,
        allow_writing_files=False,
    )


# Display prefix, ModelType, builder
MODEL_SPECS = [
    ("Logistic Regression", "LogisticRegression", lambda y: make_logreg()),
    ("Decision Tree", "DecisionTree", lambda y: make_decision_tree()),
    ("RF", "RandomForest", lambda y: make_random_forest()),
    ("LightGBM", "LightGBM", lambda y: make_lightgbm()),
    ("XGBoost", "XGBoost", lambda y: make_xgboost(y)),
    ("CatBoost", "CatBoost", lambda y: make_catboost()),
]


## Optuna

TPE search maximizing **ROC-AUC** on the last `TUNE_VALID_FRAC` of the 80% train split (still before the holdout). Random Forest uses only the most recent `TUNE_MAX_ROWS` rows for inner training. The holdout is untouched until `evaluate()`.


In [ ]:
import optuna
from optuna.samplers import TPESampler

optuna.logging.set_verbosity(optuna.logging.WARNING)

USE_OPTUNA = True
N_TRIALS = 20
TUNE_VALID_FRAC = 0.2
TUNE_MAX_ROWS = {
    "RandomForest": 100_000,
}


def _scale_pos_weight(y):
    n_neg = int((y == 0).sum())
    n_pos = max(int((y == 1).sum()), 1)
    return n_neg / n_pos


def sample_params(trial, model_type):
    if model_type == "LogisticRegression":
        return {"C": trial.suggest_float("C", 1e-3, 10.0, log=True)}
    if model_type == "DecisionTree":
        return {
            "max_depth": trial.suggest_int("max_depth", 4, 32),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 64),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 40),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        }
    if model_type == "RandomForest":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 100, 400, step=50),
            "max_depth": trial.suggest_int("max_depth", 8, 32),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        }
    if model_type == "LightGBM":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 200, 800, step=50),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 16, 96),
            "max_depth": trial.suggest_int("max_depth", 4, 12),
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 80),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        }
    if model_type == "XGBoost":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 200, 800, step=50),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 10.0),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        }
    if model_type == "CatBoost":
        return {
            "iterations": trial.suggest_int("iterations", 200, 800, step=50),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "depth": trial.suggest_int("depth", 4, 10),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
        }
    raise ValueError(f"No search space for {model_type}")


def build_model(model_type, params, y_train, n_jobs=-1):
    p = dict(params)
    if model_type == "LogisticRegression":
        return Pipeline(
            [
                ("scaler", StandardScaler()),
                (
                    "clf",
                    LogisticRegression(
                        class_weight="balanced",
                        max_iter=2000,
                        solver="lbfgs",
                        random_state=RANDOM_SEED,
                        **p,
                    ),
                ),
            ]
        )
    if model_type == "DecisionTree":
        return DecisionTreeClassifier(
            class_weight="balanced", random_state=RANDOM_SEED, **p
        )
    if model_type == "RandomForest":
        return RandomForestClassifier(
            class_weight="balanced", random_state=RANDOM_SEED, n_jobs=n_jobs, **p
        )
    if model_type == "LightGBM":
        return LGBMClassifier(
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=n_jobs,
            verbosity=-1,
            subsample_freq=1,
            **p,
        )
    if model_type == "XGBoost":
        return XGBClassifier(
            random_state=RANDOM_SEED,
            n_jobs=n_jobs,
            eval_metric="logloss",
            tree_method="hist",
            scale_pos_weight=_scale_pos_weight(y_train),
            **p,
        )
    if model_type == "CatBoost":
        return CatBoostClassifier(
            random_seed=RANDOM_SEED,
            auto_class_weights="Balanced",
            verbose=False,
            allow_writing_files=False,
            thread_count=n_jobs if n_jobs > 0 else -1,
            **p,
        )
    raise ValueError(f"Unknown model_type {model_type}")


def _inner_split(X, y, model_type):
    n = len(X)
    split = int(n * (1.0 - TUNE_VALID_FRAC))
    X_tr, X_va = X.iloc[:split], X.iloc[split:]
    y_tr, y_va = y.iloc[:split], y.iloc[split:]
    max_rows = TUNE_MAX_ROWS.get(model_type)
    if max_rows is not None and len(X_tr) > max_rows:
        X_tr = X_tr.iloc[-max_rows:]
        y_tr = y_tr.iloc[-max_rows:]
    return X_tr, X_va, y_tr, y_va


def _params_json(params):
    out = {}
    for key, value in params.items():
        if isinstance(value, (np.floating, np.integer)):
            value = value.item()
        out[key] = value
    return json.dumps(out)


def tune_hyperparams(model_type, X_train, y_train, n_trials=N_TRIALS):
    X_tr, X_va, y_tr, y_va = _inner_split(X_train, y_train, model_type)
    print(
        f"  Optuna {model_type}: {n_trials} trials  "
        f"inner train {len(X_tr):,}  inner valid {len(X_va):,}"
    )

    def objective(trial):
        params = sample_params(trial, model_type)
        model = build_model(model_type, params, y_tr, n_jobs=1)
        model.fit(X_tr, y_tr)
        y_score = positive_scores(model, X_va)
        auc = float(roc_auc_score(y_va, y_score))
        del model
        gc.collect()
        return auc

    study = optuna.create_study(
        direction="maximize",
        sampler=TPESampler(seed=RANDOM_SEED),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    best_params = dict(study.best_params)
    print(f"  best inner ROC-AUC {study.best_value:.4f}  {best_params}")
    return best_params, float(study.best_value)


print(
    f"Optuna {optuna.__version__}  USE_OPTUNA={USE_OPTUNA}  N_TRIALS={N_TRIALS}  "
    f"inner valid frac={TUNE_VALID_FRAC}"
)


In [6]:
train = train.sort_values("TransactionDT").reset_index(drop=True)
y = train["isFraud"]
split_idx = int(len(train) * 0.8)
y_train = y.iloc[:split_idx]
y_valid = y.iloc[split_idx:]

ID_COLS = ["isFraud", "TransactionID"]
FE_COLS = ["uid", "uid2", "DT_month", "DT_week", "DT_day", "DT_weekday", "DT_hour"]

baseline_cols = [c for c in train.columns if c not in ID_COLS + FE_COLS]
feature_cols = [c for c in train.columns if c not in ID_COLS]

X_train_baseline = train[baseline_cols].iloc[:split_idx]
X_valid_baseline = train[baseline_cols].iloc[split_idx:]
X_train_features = train[feature_cols].iloc[:split_idx]
X_valid_features = train[feature_cols].iloc[split_idx:]

experiments = [
    ("Baseline", X_train_baseline, X_valid_baseline),
    ("Feature Engineering", X_train_features, X_valid_features),
]

reduced_path = DATASET_PATH / "merged_train_reduced.parquet"
if not reduced_path.exists():
    raise FileNotFoundError(f"{reduced_path} missing — run 01_clean_dataset.ipynb")

train_reduced = pd.read_parquet(reduced_path).sort_values("TransactionDT").reset_index(drop=True)
if len(train_reduced) != len(train):
    raise ValueError(f"reduced rows {len(train_reduced):,} != full train {len(train):,}")

reduced_baseline_cols = [c for c in train_reduced.columns if c not in ID_COLS + FE_COLS]
reduced_feature_cols = [c for c in train_reduced.columns if c not in ID_COLS]
X_train_reduced_baseline = train_reduced[reduced_baseline_cols].iloc[:split_idx]
X_valid_reduced_baseline = train_reduced[reduced_baseline_cols].iloc[split_idx:]
X_train_reduced_features = train_reduced[reduced_feature_cols].iloc[:split_idx]
X_valid_reduced_features = train_reduced[reduced_feature_cols].iloc[split_idx:]

experiments.extend(
    [
        ("Reduced Baseline", X_train_reduced_baseline, X_valid_reduced_baseline),
        ("Reduced Feature Engineering", X_train_reduced_features, X_valid_reduced_features),
    ]
)

for label, X_tr, X_va in experiments:
    print(f"{label}: {X_tr.shape[1]} features  train {len(X_tr):,}  valid {len(X_va):,}")


Baseline: 432 features  train 472,432  valid 118,108
Feature Engineering: 439 features  train 472,432  valid 118,108
Reduced Baseline: 342 features  train 472,432  valid 118,108
Reduced Feature Engineering: 348 features  train 472,432  valid 118,108


In [7]:
all_results = []

RESULT_COLS = [
    "Model",
    "ModelType",
    "Dataset",
    "Features",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC",
    "PR-AUC",
    "Balanced Accuracy",
    "MCC",
    "TN",
    "FP",
    "FN",
    "TP",
    "TuneROC-AUC",
    "BestParams",
    "Top20Importances",
]
CORE_SUFFIXES = (
    " - Baseline",
    " - Feature Engineering",
    " - Reduced Baseline",
    " - Reduced Feature Engineering",
)


def _is_core_run(name) -> bool:
    text = str(name)
    return any(text.endswith(suffix) for suffix in CORE_SUFFIXES)


def save_results():
    if not all_results:
        return
    new_df = pd.DataFrame(all_results).drop_duplicates(subset=["Model"], keep="last")
    if "Dataset" not in new_df.columns:
        new_df["Dataset"] = new_df["Model"].astype(str).str.split(" - ", n=1).str[1]
    new_df = new_df[[c for c in RESULT_COLS if c in new_df.columns]]
    path = SAVED_PATH / "ml_results.parquet"
    if path.exists():
        old = pd.read_parquet(path)
        if "Model" in old.columns:
            old = old[old["Model"].map(_is_core_run)]
            old = old[~old["Model"].astype(str).str.startswith(("SVM -", "KNN -"))]
            if "ModelType" in old.columns:
                old = old[~old["ModelType"].isin(["SVM", "KNN"])]
            old = old[~old["Model"].isin(new_df["Model"])]
            if "Dataset" not in old.columns:
                old["Dataset"] = old["Model"].astype(str).str.split(" - ", n=1).str[1]
            old = old[[c for c in RESULT_COLS if c in old.columns]]
        new_df = pd.concat([old, new_df], ignore_index=True, sort=False)
    new_df = new_df.drop_duplicates(subset=["Model"], keep="last")
    new_df.to_parquet(path, index=False)
    print(f"saved {path}  n={len(new_df)}")


def run_family(prefix, model_type, factory):
    """Tune (optional), fit on all four datasets, and save after each."""
    for label, X_tr, X_va in experiments:
        name = f"{prefix} - {label}"
        best_params = {}
        tune_auc = None
        if USE_OPTUNA and N_TRIALS > 0:
            best_params, tune_auc = tune_hyperparams(model_type, X_tr, y_train)
            model = build_model(model_type, best_params, y_train)
        else:
            model = factory(y_train)
        row = evaluate(model, X_tr, X_va, y_train, y_valid, name)
        row["ModelType"] = model_type
        row["Dataset"] = label
        row["BestParams"] = _params_json(best_params)
        row["TuneROC-AUC"] = tune_auc
        all_results.append(row)
        save_results()
        gc.collect()


## Fit

One cell per model. Each cell tunes that model on the **four** datasets (`N_TRIALS` Optuna trials, inner temporal split), refits the best params on the full train split, scores the holdout, and writes `ml_results.parquet`. Run them in order (or skip a cell if that family is already saved). Set `USE_OPTUNA = False` to use the default factories only.


### Logistic Regression


In [ ]:
run_family("Logistic Regression", "LogisticRegression", lambda y: make_logreg())

Logistic Regression - Baseline
Features: 432


### Decision Tree


In [ ]:
run_family("Decision Tree", "DecisionTree", lambda y: make_decision_tree())

### Random Forest


In [ ]:
run_family("RF", "RandomForest", lambda y: make_random_forest())

### LightGBM


In [ ]:
run_family("LightGBM", "LightGBM", lambda y: make_lightgbm())

### XGBoost


In [ ]:
run_family("XGBoost", "XGBoost", make_xgboost)

### CatBoost


In [ ]:
run_family("CatBoost", "CatBoost", lambda y: make_catboost())